# 07 — Context-Adaptive Gated Fusion Model

**Objective:** Develop and validate the context-aware gated fusion architecture that combines text and contextual features through a learned, context-adaptive gating mechanism.

**Key capabilities tested:**
- Gated fusion of text + context representations
- Graceful fallback to text-only when context is missing (`context_mask = 0`)
- Gate behavior analysis under different context availability conditions

**Note:** This notebook uses synthetic data for architecture validation until Members 1–2 complete the feature engineering pipeline. All synthetic results are clearly marked as **ARCHITECTURE SANITY CHECKS — NOT FINAL RESEARCH RESULTS**.

---
## 1. Imports

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## 2. Project Path Setup

In [ ]:
# Determine project root — works whether running from notebooks/ or project root
NOTEBOOK_DIR = Path(os.getcwd())
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Add project root to sys.path so we can import from src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Key directories
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
RESULTS_METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
RESULTS_MODELS_DIR = PROJECT_ROOT / "results" / "models"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data (processed): {DATA_PROCESSED_DIR}")
print(f"Results (figures): {RESULTS_FIGURES_DIR}")

---
## 3. Import Member 4 Model Modules

In [ ]:
from src.models.gated_fusion import GatedFusionModel
from src.models.domain_alignment import DomainAlignment, domain_alignment_loss

print("Member 4 modules imported successfully.")

---
## 4. Configuration & Reproducibility

In [ ]:
# ── Reproducibility ────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Hyperparameters (synthetic sanity check) ───────────────────────
# These will be updated when real feature dimensions are known.
CONFIG = {
    "text_dim": 768,       # Placeholder — depends on Member 2's text features
    "context_dim": 32,     # Placeholder — depends on Member 2's context features
    "hidden_dim": 128,
    "num_classes": 2,
    "dropout": 0.3,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "num_epochs": 20,
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

---
## 5. Load Prepared Feature Data

When Members 1–2 have completed their feature engineering pipeline, load real features here from `data/processed/`.

**Expected interface:**
- `text_features`: `(num_samples, text_dim)` — from Member 2's text feature extractor
- `context_features`: `(num_samples, context_dim)` — from Member 2's context feature extractor
- `context_mask`: `(num_samples,)` — 1 = context available, 0 = missing
- `labels`: `(num_samples,)` — target class labels

In [ ]:
# ── Check for real processed data ─────────────────────────────────
REAL_DATA_AVAILABLE = False

# Candidate file paths — update these when the actual filenames are known
TEXT_FEATURES_PATH = DATA_PROCESSED_DIR / "text_features.npy"
CONTEXT_FEATURES_PATH = DATA_PROCESSED_DIR / "context_features.npy"
CONTEXT_MASK_PATH = DATA_PROCESSED_DIR / "context_mask.npy"
LABELS_PATH = DATA_PROCESSED_DIR / "labels.npy"

if all(p.exists() for p in [TEXT_FEATURES_PATH, CONTEXT_FEATURES_PATH,
                             CONTEXT_MASK_PATH, LABELS_PATH]):
    REAL_DATA_AVAILABLE = True
    text_features_np = np.load(TEXT_FEATURES_PATH)
    context_features_np = np.load(CONTEXT_FEATURES_PATH)
    context_mask_np = np.load(CONTEXT_MASK_PATH)
    labels_np = np.load(LABELS_PATH)
    
    # Update config with real dimensions
    CONFIG["text_dim"] = text_features_np.shape[1]
    CONFIG["context_dim"] = context_features_np.shape[1]
    CONFIG["num_classes"] = len(np.unique(labels_np))
    
    print(f"Loaded REAL data:")
    print(f"  text_features:    {text_features_np.shape}")
    print(f"  context_features: {context_features_np.shape}")
    print(f"  context_mask:     {context_mask_np.shape}")
    print(f"  labels:           {labels_np.shape}")
    print(f"  Updated config: text_dim={CONFIG['text_dim']}, "
          f"context_dim={CONFIG['context_dim']}, num_classes={CONFIG['num_classes']}")
else:
    print("Real processed data NOT found. Will use synthetic data for architecture validation.")
    print("(Awaiting Members 1–2 feature engineering pipeline)")

---
## ⚠️ ARCHITECTURE SANITY CHECK — NOT FINAL RESEARCH RESULTS

The following sections use **synthetic data** to validate that the model architecture is correct, the gating mechanism works, and missing context is handled properly.

**These results must NOT be interpreted as research findings.**

When real features from Members 1–2 become available, re-run this notebook with `REAL_DATA_AVAILABLE = True` (the loading cell above will detect real data automatically).

---
## 6. Generate Synthetic Data (Architecture Validation Only)

In [ ]:
if not REAL_DATA_AVAILABLE:
    print("Generating synthetic data for architecture sanity check...")
    
    NUM_SAMPLES = 2000
    CONTEXT_MISSING_RATIO = 0.3  # 30% of samples have no context
    
    np.random.seed(SEED)
    
    # Synthetic features
    text_features_np = np.random.randn(NUM_SAMPLES, CONFIG["text_dim"]).astype(np.float32)
    context_features_np = np.random.randn(NUM_SAMPLES, CONFIG["context_dim"]).astype(np.float32)
    
    # Synthetic context mask — 70% have context, 30% do not
    context_mask_np = np.ones(NUM_SAMPLES, dtype=np.float32)
    missing_indices = np.random.choice(
        NUM_SAMPLES, size=int(NUM_SAMPLES * CONTEXT_MISSING_RATIO), replace=False
    )
    context_mask_np[missing_indices] = 0.0
    
    # Synthetic binary labels
    labels_np = np.random.randint(0, CONFIG["num_classes"], size=NUM_SAMPLES).astype(np.int64)
    
    print(f"  Samples: {NUM_SAMPLES}")
    print(f"  Text features shape: {text_features_np.shape}")
    print(f"  Context features shape: {context_features_np.shape}")
    print(f"  Context available: {int(context_mask_np.sum())} / {NUM_SAMPLES} "
          f"({context_mask_np.mean()*100:.0f}%)")
    print(f"  Label distribution: {dict(zip(*np.unique(labels_np, return_counts=True)))}")
else:
    print("Using real data — skipping synthetic generation.")

---
## 7. Validate Input Dimensions

In [ ]:
assert text_features_np.ndim == 2, f"text_features must be 2D, got {text_features_np.ndim}D"
assert context_features_np.ndim == 2, f"context_features must be 2D, got {context_features_np.ndim}D"
assert text_features_np.shape[0] == context_features_np.shape[0], "Sample count mismatch"
assert context_mask_np.shape[0] == text_features_np.shape[0], "Mask length mismatch"
assert labels_np.shape[0] == text_features_np.shape[0], "Labels length mismatch"
assert set(np.unique(context_mask_np)).issubset({0.0, 1.0}), "Mask must contain only 0 and 1"

print("All input dimension checks passed.")
print(f"  text_dim:    {text_features_np.shape[1]}")
print(f"  context_dim: {context_features_np.shape[1]}")
print(f"  num_samples: {text_features_np.shape[0]}")
print(f"  num_classes: {len(np.unique(labels_np))}")

---
## 8. Inspect Context Availability

In [ ]:
n_available = int(context_mask_np.sum())
n_missing = len(context_mask_np) - n_available

print(f"Context availability:")
print(f"  Available: {n_available} ({n_available/len(context_mask_np)*100:.1f}%)")
print(f"  Missing:   {n_missing} ({n_missing/len(context_mask_np)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Context Available", "Context Missing"], [n_available, n_missing],
       color=["#2ecc71", "#e74c3c"])
ax.set_ylabel("Number of Samples")
ax.set_title("Context Availability Distribution")
for i, v in enumerate([n_available, n_missing]):
    ax.text(i, v + 10, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Prepare Data Splits & DataLoaders

In [ ]:
# ── Train / Validation / Test split (60/20/20) ────────────────────
indices = np.arange(len(labels_np))

train_idx, temp_idx = train_test_split(indices, test_size=0.4, random_state=SEED,
                                       stratify=labels_np)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED,
                                     stratify=labels_np[temp_idx])

print(f"Split sizes — Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")


def make_dataloader(idx, batch_size, shuffle=True):
    """Create a DataLoader from index array."""
    dataset = TensorDataset(
        torch.tensor(text_features_np[idx]),
        torch.tensor(context_features_np[idx]),
        torch.tensor(context_mask_np[idx]),
        torch.tensor(labels_np[idx]),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader = make_dataloader(train_idx, CONFIG["batch_size"], shuffle=True)
val_loader = make_dataloader(val_idx, CONFIG["batch_size"], shuffle=False)
test_loader = make_dataloader(test_idx, CONFIG["batch_size"], shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, "
      f"Test batches: {len(test_loader)}")

---
## 10. Initialize Gated Fusion Model

In [ ]:
model = GatedFusionModel(
    text_dim=CONFIG["text_dim"],
    context_dim=CONFIG["context_dim"],
    hidden_dim=CONFIG["hidden_dim"],
    num_classes=CONFIG["num_classes"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

print("Model architecture:")
print(model)
print(f"\nModel config: {model.get_config()}")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

---
## 11. Train Model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])

train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(CONFIG["num_epochs"]):
    # ── Training ──────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    for text_b, ctx_b, mask_b, labels_b in train_loader:
        text_b = text_b.to(DEVICE)
        ctx_b = ctx_b.to(DEVICE)
        mask_b = mask_b.to(DEVICE)
        labels_b = labels_b.to(DEVICE)
        
        optimizer.zero_grad()
        logits, gate_vals = model(text_b, ctx_b, mask_b)
        loss = criterion(logits, labels_b)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)
    
    # ── Validation ────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels = []
    val_batches = 0
    
    with torch.no_grad():
        for text_b, ctx_b, mask_b, labels_b in val_loader:
            text_b = text_b.to(DEVICE)
            ctx_b = ctx_b.to(DEVICE)
            mask_b = mask_b.to(DEVICE)
            labels_b = labels_b.to(DEVICE)
            
            logits, _ = model(text_b, ctx_b, mask_b)
            loss = criterion(logits, labels_b)
            val_loss += loss.item()
            val_batches += 1
            
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())
            val_labels.extend(labels_b.cpu().numpy())
    
    avg_val_loss = val_loss / val_batches
    val_acc = accuracy_score(val_labels, val_preds)
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:2d}/{CONFIG['num_epochs']}]  "
              f"Train Loss: {avg_train_loss:.4f}  "
              f"Val Loss: {avg_val_loss:.4f}  "
              f"Val Acc: {val_acc:.4f}")

---
## 12. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(range(1, CONFIG["num_epochs"] + 1), train_losses, label="Train Loss", marker=".")
axes[0].plot(range(1, CONFIG["num_epochs"] + 1), val_losses, label="Val Loss", marker=".")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation accuracy
axes[1].plot(range(1, CONFIG["num_epochs"] + 1), val_accuracies, label="Val Accuracy",
             marker=".", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("ARCHITECTURE SANITY CHECK — NOT FINAL RESULTS", fontsize=10,
             color="red", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 13. Evaluate on Test Set

In [ ]:
model.eval()
all_preds = []
all_labels = []
all_probs = []
all_gates = []
all_masks = []

with torch.no_grad():
    for text_b, ctx_b, mask_b, labels_b in test_loader:
        text_b = text_b.to(DEVICE)
        ctx_b = ctx_b.to(DEVICE)
        mask_b = mask_b.to(DEVICE)
        
        logits, gate_vals = model(text_b, ctx_b, mask_b)
        probs = torch.softmax(logits, dim=1)
        
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels_b.numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # Probability of positive class
        all_gates.extend(gate_vals.squeeze().cpu().numpy())
        all_masks.extend(mask_b.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_gates = np.array(all_gates)
all_masks = np.array(all_masks)

print("Test Set Results (SANITY CHECK — synthetic data):")
print("=" * 50)
print(classification_report(all_labels, all_preds, digits=4))

---
## 14. Analyze Gate Behavior

The gate controls how much context vs. text influences the final prediction:
- `gate ≈ 0` → model relies on **text only**
- `gate ≈ 1` → model relies on **context only**
- When `context_mask = 0`, gate is **forced to 0** (text-only fallback)

In [ ]:
# Split gate values by context availability
gates_ctx_available = all_gates[all_masks == 1]
gates_ctx_missing = all_gates[all_masks == 0]

print(f"Gate statistics (context AVAILABLE):")
print(f"  Mean: {gates_ctx_available.mean():.4f}")
print(f"  Std:  {gates_ctx_available.std():.4f}")
print(f"  Min:  {gates_ctx_available.min():.4f}")
print(f"  Max:  {gates_ctx_available.max():.4f}")

print(f"\nGate statistics (context MISSING):")
print(f"  Mean: {gates_ctx_missing.mean():.4f}")
print(f"  Std:  {gates_ctx_missing.std():.4f}")
print(f"  All zero: {np.allclose(gates_ctx_missing, 0.0)}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(gates_ctx_available, bins=30, color="#3498db", alpha=0.8, edgecolor="white")
axes[0].set_xlabel("Gate Value")
axes[0].set_ylabel("Count")
axes[0].set_title("Gate Distribution (Context Available)")
axes[0].axvline(x=gates_ctx_available.mean(), color="red", linestyle="--",
                label=f"Mean={gates_ctx_available.mean():.3f}")
axes[0].legend()

axes[1].hist(gates_ctx_missing, bins=30, color="#e74c3c", alpha=0.8, edgecolor="white")
axes[1].set_xlabel("Gate Value")
axes[1].set_ylabel("Count")
axes[1].set_title("Gate Distribution (Context Missing)")
axes[1].set_xlim(-0.1, 1.1)

plt.suptitle("ARCHITECTURE SANITY CHECK — NOT FINAL RESULTS", fontsize=10,
             color="red", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 15. Missing-Context Experiment

Compare model performance on:
- **Experiment A:** Samples where context IS available (`context_mask = 1`)
- **Experiment B:** Samples where context is MISSING (`context_mask = 0`)

In [ ]:
def compute_metrics(y_true, y_pred, y_prob=None):
    """
    Compute standard classification metrics.
    
    This is a lightweight helper for development. When Member 3's
    src/evaluation/metrics.py is available, replace with their implementation.
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if y_prob is not None and len(np.unique(y_true)) == 2:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
        except ValueError:
            metrics["roc_auc"] = float("nan")
    return metrics


# Split test results by context availability
ctx_avail_mask = all_masks == 1
ctx_miss_mask = all_masks == 0

print("Experiment A — Context Available:")
print(f"  Samples: {ctx_avail_mask.sum()}")
metrics_a = compute_metrics(all_labels[ctx_avail_mask], all_preds[ctx_avail_mask],
                            all_probs[ctx_avail_mask])
for k, v in metrics_a.items():
    print(f"  {k}: {v:.4f}")

print(f"\nExperiment B — Context Missing:")
print(f"  Samples: {ctx_miss_mask.sum()}")
metrics_b = compute_metrics(all_labels[ctx_miss_mask], all_preds[ctx_miss_mask],
                            all_probs[ctx_miss_mask])
for k, v in metrics_b.items():
    print(f"  {k}: {v:.4f}")

print("\n⚠️  SANITY CHECK ONLY — these numbers reflect synthetic data.")

In [ ]:
# ── Visual comparison ─────────────────────────────────────────────
metric_names = list(metrics_a.keys())
values_a = [metrics_a[m] for m in metric_names]
values_b = [metrics_b[m] for m in metric_names]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_a = ax.bar(x - width/2, values_a, width, label="Context Available", color="#2ecc71")
bars_b = ax.bar(x + width/2, values_b, width, label="Context Missing", color="#e74c3c")

ax.set_ylabel("Score")
ax.set_title("Missing-Context Experiment — SANITY CHECK (Synthetic Data)")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)

# Value labels
for bar in bars_a:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", fontsize=8)
for bar in bars_b:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

---
## 16. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=[f"Class {i}" for i in range(CONFIG["num_classes"])],
            yticklabels=[f"Class {i}" for i in range(CONFIG["num_classes"])])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — SANITY CHECK (Synthetic Data)")
plt.tight_layout()
plt.show()

---
## 17. Save Outputs

**Important:** Since this notebook uses synthetic data, outputs are saved with `_sanity_check` suffix to prevent confusion with real research results. When real data is used, remove the suffix.

In [ ]:
if not REAL_DATA_AVAILABLE:
    print("Synthetic data mode — skipping saves to results/ to avoid polluting ")
    print("the research results directory with non-research outputs.")
    print("\nRe-run this notebook with real data to save actual results.")
else:
    # Save only when using real data
    # Uncomment and adjust when integrating real features:
    #
    # torch.save(model.state_dict(), RESULTS_MODELS_DIR / "gated_fusion_context_model.pt")
    # print(f"Model saved to {RESULTS_MODELS_DIR / 'gated_fusion_context_model.pt'}")
    #
    # metrics_df = pd.DataFrame({"Metric": list(metrics_a.keys()),
    #                            "Context_Available": list(metrics_a.values()),
    #                            "Context_Missing": list(metrics_b.values())})
    # metrics_df.to_csv(RESULTS_METRICS_DIR / "context_model_metrics.csv", index=False)
    # print(f"Metrics saved to {RESULTS_METRICS_DIR / 'context_model_metrics.csv'}")
    pass

---
## 18. Conclusions & Observations

### Architecture Validation Results

1. **Model instantiation:** GatedFusionModel successfully initializes with configurable dimensions.
2. **Forward pass:** Produces logits of correct shape `(batch_size, num_classes)` and gate values `(batch_size, 1)`.
3. **Context masking:** When `context_mask = 0`, gate values are correctly forced to `0.0`, confirming text-only fallback.
4. **Training loop:** Model trains and backpropagates without errors.
5. **Gate behavior:** Gate values for context-available samples are learned; gate values for context-missing samples remain zero.

### Awaiting Integration

- **Members 1–2:** Real `text_features`, `context_features`, `context_mask`, and `labels` from the feature engineering pipeline.
- **Member 3:** Evaluation utilities (`src/evaluation/metrics.py`, `src/evaluation/plots.py`) for standardized metric computation.

### Next Steps

- Once real features are available, re-run this notebook with `REAL_DATA_AVAILABLE = True`.
- Proceed to `08_final_model.ipynb` for complete experiments including domain alignment.